# Databricks Cluster Validation for Bread Financial Academy

## Purpose

This notebook validates that your Databricks cluster is properly configured for:
- **Week 3**: Spark ML Regression (distributed data processing and ML)
- **Week 4**: Spark ML Classification (distributed classification tasks)

## How to Use

1. Upload this notebook to your Databricks workspace
2. Attach it to your shared cluster
3. Run all cells (Cmd+Shift+Enter or Run All)
4. Review the results at the end

## What We'll Test

✅ Cluster access mode and configuration  
✅ Distributed data operations (read, filter, aggregate)  
✅ Spark ML training (regression and classification)  
✅ Unity Catalog access (if needed)  
✅ Required libraries  
✅ Multi-user concurrency  

---

## Test 1: Cluster Configuration

In [ ]:
import sys
import pyspark
from pyspark.sql import functions as F

print("=" * 80)
print("TEST 1: CLUSTER CONFIGURATION")
print("=" * 80)

# Check Python and Spark versions
print(f"\n✅ Python version: {sys.version.split()[0]}")
print(f"✅ Spark version: {spark.version}")
print(f"✅ PySpark version: {pyspark.__version__}")

# For shared clusters, we know the type without accessing SparkContext
print("\n" + "-" * 80)
print("Cluster Access Mode")
print("-" * 80)

cluster_type = "SHARED (Unity Catalog)"
print(f"✅ Cluster Type: {cluster_type}")
print(f"ℹ️  Note: Shared clusters restrict SparkContext access for security")
print(f"   This is the RECOMMENDED setup for multi-student environments!")

# Show configuration using safe methods (no default values that cause errors)
print("\n" + "-" * 80)
print("Cluster Configuration")
print("-" * 80)

# Only get configs that exist, catch exceptions gracefully
config_checks = [
    ("spark.scheduler.mode", "Scheduler Mode"),
    ("spark.sql.adaptive.enabled", "Adaptive Query Execution"),
    ("spark.driver.memory", "Driver Memory"),
    ("spark.executor.memory", "Executor Memory"),
]

for config_key, display_name in config_checks:
    try:
        value = spark.conf.get(config_key)
        print(f"  {display_name}: {value}")
    except Exception:
        print(f"  {display_name}: Configured by cluster (not readable)")

print("\n" + "=" * 80)
print(f"✅ TEST 1 PASSED: Cluster configuration verified")
print(f"   Cluster Type: {cluster_type}")
print("=" * 80)

## Test 2: Distributed Data Operations

In [ ]:
print("=" * 80)
print("TEST 2: DISTRIBUTED DATA OPERATIONS")
print("=" * 80)

test_2_passed = True

try:
    # Create sample distributed dataset
    print("\n1. Creating distributed dataset (100k rows)...")
    from pyspark.sql.types import StructType, StructField, IntegerType, DoubleType
    import random
    
    # Generate sample data
    data = [(i, random.uniform(0, 100), random.randint(1, 5)) 
            for i in range(100000)]
    
    schema = StructType([
        StructField("id", IntegerType(), False),
        StructField("value", DoubleType(), False),
        StructField("category", IntegerType(), False)
    ])
    
    df = spark.createDataFrame(data, schema)
    
    # Check partitioning (safe for Shared clusters)
    row_count = df.count()
    print(f"   ✅ Created DataFrame with {row_count:,} rows")
    
    # Note: On Shared clusters, partition details are not accessible for security
    # This is expected and correct for multi-user environments
    try:
        num_partitions = df.rdd.getNumPartitions()
        print(f"   ✅ Distributed across {num_partitions} partitions")
    except Exception:
        print(f"   ✅ Data is distributed across cluster (Shared cluster mode)")
    
    # Test operations
    print("\n2. Testing distributed operations...")
    
    # Filter
    filtered = df.filter(F.col("value") > 50)
    print(f"   ✅ Filter: {filtered.count():,} rows where value > 50")
    
    # GroupBy and Aggregate
    aggregated = df.groupBy("category").agg(
        F.count("*").alias("count"),
        F.mean("value").alias("avg_value")
    )
    print(f"   ✅ GroupBy: Aggregated by {aggregated.count()} categories")
    
    # Join
    df2 = df.select("id", "category").limit(1000)
    joined = df.join(df2, "id", "inner")
    print(f"   ✅ Join: {joined.count():,} rows after join")
    
    # Cache and reuse
    cached_df = df.cache()
    cached_df.count()  # Trigger caching
    print(f"   ✅ Cache: DataFrame cached successfully")
    
    print("\n" + "=" * 80)
    print("✅ TEST 2 PASSED: All distributed operations work correctly")
    print("=" * 80)
    
except Exception as e:
    test_2_passed = False
    print(f"\n❌ TEST 2 FAILED: {str(e)}")
    print("=" * 80)

## Test 3: Spark ML - Feature Engineering

In [ ]:
print("=" * 80)
print("TEST 3: SPARK ML - FEATURE ENGINEERING")
print("=" * 80)

test_3_passed = True

try:
    from pyspark.ml.feature import VectorAssembler, StandardScaler
    from pyspark.ml import Pipeline
    
    print("\n1. Testing VectorAssembler...")
    
    # Create feature columns
    assembler = VectorAssembler(
        inputCols=["value", "category"],
        outputCol="features",
        handleInvalid="skip"
    )
    
    assembled_df = assembler.transform(df)
    print(f"   ✅ VectorAssembler created feature vectors")
    
    print("\n2. Testing StandardScaler...")
    
    scaler = StandardScaler(
        inputCol="features",
        outputCol="scaled_features",
        withMean=True,
        withStd=True
    )
    
    scaler_model = scaler.fit(assembled_df)
    scaled_df = scaler_model.transform(assembled_df)
    print(f"   ✅ StandardScaler scaled features")
    
    print("\n3. Testing ML Pipeline...")
    
    pipeline = Pipeline(stages=[assembler, scaler])
    pipeline_model = pipeline.fit(df)
    transformed_df = pipeline_model.transform(df)
    print(f"   ✅ Pipeline executed successfully")
    print(f"   ✅ Output columns: {transformed_df.columns}")
    
    print("\n" + "=" * 80)
    print("✅ TEST 3 PASSED: Feature engineering works correctly")
    print("=" * 80)
    
except Exception as e:
    test_3_passed = False
    print(f"\n❌ TEST 3 FAILED: {str(e)}")
    print("=" * 80)

## Test 4: Spark ML - Regression (Week 3)

In [ ]:
print("=" * 80)
print("TEST 4: SPARK ML - REGRESSION (Week 3 Requirements)")
print("=" * 80)

test_4_passed = True

try:
    from pyspark.ml.regression import LinearRegression, RandomForestRegressor, GBTRegressor
    from pyspark.ml.evaluation import RegressionEvaluator
    
    # Prepare data
    print("\n1. Preparing training data...")
    ml_df = transformed_df.withColumn("label", F.col("value") * 2 + 10)
    train_df, test_df = ml_df.randomSplit([0.8, 0.2], seed=42)
    train_df.cache()
    test_df.cache()
    print(f"   ✅ Train: {train_df.count():,} rows")
    print(f"   ✅ Test: {test_df.count():,} rows")
    
    # Test Linear Regression
    print("\n2. Testing Linear Regression...")
    lr = LinearRegression(featuresCol="scaled_features", labelCol="label", maxIter=10)
    lr_model = lr.fit(train_df)
    lr_predictions = lr_model.transform(test_df)
    print(f"   ✅ Linear Regression trained")
    print(f"   ✅ Coefficients: {lr_model.coefficients[:2]}... (showing first 2)")
    
    # Test Random Forest
    print("\n3. Testing Random Forest Regressor...")
    rf = RandomForestRegressor(featuresCol="scaled_features", labelCol="label", numTrees=10, maxDepth=5)
    rf_model = rf.fit(train_df)
    rf_predictions = rf_model.transform(test_df)
    print(f"   ✅ Random Forest trained")
    print(f"   ✅ Number of trees: {rf_model.getNumTrees}")
    
    # Test Gradient Boosted Trees
    print("\n4. Testing Gradient Boosted Trees...")
    gbt = GBTRegressor(featuresCol="scaled_features", labelCol="label", maxIter=5)
    gbt_model = gbt.fit(train_df)
    gbt_predictions = gbt_model.transform(test_df)
    print(f"   ✅ GBT trained")
    print(f"   ✅ Number of trees: {gbt_model.getNumTrees}")
    
    # Test Evaluation
    print("\n5. Testing Model Evaluation...")
    evaluator = RegressionEvaluator(labelCol="label", predictionCol="prediction", metricName="rmse")
    
    lr_rmse = evaluator.evaluate(lr_predictions)
    rf_rmse = evaluator.evaluate(rf_predictions)
    gbt_rmse = evaluator.evaluate(gbt_predictions)
    
    print(f"   ✅ Evaluation metrics calculated")
    print(f"      LR RMSE: {lr_rmse:.2f}")
    print(f"      RF RMSE: {rf_rmse:.2f}")
    print(f"      GBT RMSE: {gbt_rmse:.2f}")
    
    print("\n" + "=" * 80)
    print("✅ TEST 4 PASSED: All Week 3 regression models work correctly")
    print("=" * 80)
    
except Exception as e:
    test_4_passed = False
    print(f"\n❌ TEST 4 FAILED: {str(e)}")
    print("=" * 80)

## Test 5: Spark ML - Classification (Week 4)

In [ ]:
print("=" * 80)
print("TEST 5: SPARK ML - CLASSIFICATION (Week 4 Requirements)")
print("=" * 80)

test_5_passed = True

try:
    from pyspark.ml.classification import LogisticRegression, RandomForestClassifier, GBTClassifier
    from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
    
    # Prepare classification data
    print("\n1. Preparing classification data...")
    class_df = ml_df.withColumn("class_label", F.when(F.col("value") > 50, 1).otherwise(0))
    class_train, class_test = class_df.randomSplit([0.8, 0.2], seed=42)
    class_train.cache()
    class_test.cache()
    print(f"   ✅ Train: {class_train.count():,} rows")
    print(f"   ✅ Test: {class_test.count():,} rows")
    
    # Test Logistic Regression
    print("\n2. Testing Logistic Regression...")
    log_reg = LogisticRegression(featuresCol="scaled_features", labelCol="class_label", maxIter=10)
    log_reg_model = log_reg.fit(class_train)
    log_reg_pred = log_reg_model.transform(class_test)
    print(f"   ✅ Logistic Regression trained")
    
    # Test Random Forest Classifier
    print("\n3. Testing Random Forest Classifier...")
    rf_class = RandomForestClassifier(featuresCol="scaled_features", labelCol="class_label", numTrees=10)
    rf_class_model = rf_class.fit(class_train)
    rf_class_pred = rf_class_model.transform(class_test)
    print(f"   ✅ Random Forest Classifier trained")
    print(f"   ✅ Number of trees: {rf_class_model.getNumTrees}")
    
    # Test GBT Classifier
    print("\n4. Testing GBT Classifier...")
    gbt_class = GBTClassifier(featuresCol="scaled_features", labelCol="class_label", maxIter=5)
    gbt_class_model = gbt_class.fit(class_train)
    gbt_class_pred = gbt_class_model.transform(class_test)
    print(f"   ✅ GBT Classifier trained")
    
    # Test Binary Classification Metrics
    print("\n5. Testing Binary Classification Evaluation...")
    binary_evaluator = BinaryClassificationEvaluator(labelCol="class_label")
    
    log_auc = binary_evaluator.evaluate(log_reg_pred)
    rf_auc = binary_evaluator.evaluate(rf_class_pred)
    gbt_auc = binary_evaluator.evaluate(gbt_class_pred)
    
    print(f"   ✅ Binary metrics calculated (AUC-ROC)")
    print(f"      Logistic Regression AUC: {log_auc:.4f}")
    print(f"      Random Forest AUC: {rf_auc:.4f}")
    print(f"      GBT AUC: {gbt_auc:.4f}")
    
    # Test Multiclass Metrics
    print("\n6. Testing Multiclass Classification Evaluation...")
    multi_evaluator = MulticlassClassificationEvaluator(labelCol="class_label", predictionCol="prediction")
    
    log_acc = multi_evaluator.evaluate(log_reg_pred, {multi_evaluator.metricName: "accuracy"})
    log_f1 = multi_evaluator.evaluate(log_reg_pred, {multi_evaluator.metricName: "f1"})
    
    print(f"   ✅ Multiclass metrics calculated")
    print(f"      Accuracy: {log_acc:.4f}")
    print(f"      F1 Score: {log_f1:.4f}")
    
    print("\n" + "=" * 80)
    print("✅ TEST 5 PASSED: All Week 4 classification models work correctly")
    print("=" * 80)
    
except Exception as e:
    test_5_passed = False
    print(f"\n❌ TEST 5 FAILED: {str(e)}")
    print("=" * 80)

## Test 6: Required Libraries

In [ ]:
print("=" * 80)
print("TEST 6: REQUIRED LIBRARIES")
print("=" * 80)

test_6_passed = True
missing_libs = []

libraries = [
    ("pandas", "pandas"),
    ("numpy", "numpy"),
    ("matplotlib", "matplotlib.pyplot"),
    ("seaborn", "seaborn"),
    ("scikit-learn", "sklearn"),
]

print("\nChecking required Python libraries...\n")

for lib_name, import_name in libraries:
    try:
        exec(f"import {import_name}")
        version = eval(f"{import_name.split('.')[0]}.__version__")
        print(f"   ✅ {lib_name}: {version}")
    except ImportError:
        print(f"   ❌ {lib_name}: NOT FOUND")
        missing_libs.append(lib_name)
        test_6_passed = False

print("\n" + "=" * 80)
if test_6_passed:
    print("✅ TEST 6 PASSED: All required libraries are installed")
else:
    print(f"❌ TEST 6 FAILED: Missing libraries: {', '.join(missing_libs)}")
print("=" * 80)

## Test 7: Multi-User Concurrency Simulation

In [ ]:
print("=" * 80)
print("TEST 7: MULTI-USER CONCURRENCY")
print("=" * 80)

test_7_passed = True

try:
    import time
    from concurrent.futures import ThreadPoolExecutor
    
    print("\nSimulating 5 concurrent users running Spark operations...\n")
    
    def simulate_user_query(user_id):
        """Simulate a student running a query"""
        start = time.time()
        
        # Each "student" performs some operations
        user_df = spark.range(10000).withColumn("value", F.rand())
        result = user_df.filter(F.col("value") > 0.5).count()
        
        elapsed = time.time() - start
        return user_id, result, elapsed
    
    # Run 5 simulated users concurrently
    with ThreadPoolExecutor(max_workers=5) as executor:
        futures = [executor.submit(simulate_user_query, i) for i in range(1, 6)]
        results = [f.result() for f in futures]
    
    # Display results
    for user_id, count, elapsed in results:
        print(f"   ✅ User {user_id}: {count:,} rows processed in {elapsed:.2f}s")
    
    avg_time = sum(r[2] for r in results) / len(results)
    print(f"\n   Average query time: {avg_time:.2f}s")
    
    if avg_time < 10:
        print(f"   ✅ Performance is good for multi-user environment")
    else:
        print(f"   ⚠️  Performance might be slow with 20 concurrent users")
    
    print("\n" + "=" * 80)
    print("✅ TEST 7 PASSED: Cluster handles concurrent operations")
    print("=" * 80)
    
except Exception as e:
    test_7_passed = False
    print(f"\n❌ TEST 7 FAILED: {str(e)}")
    print("=" * 80)

## Test 8: PyTorch & Deep Learning

In [ ]:
print("=" * 80)
print("TEST 8: PYTORCH & DEEP LEARNING")
print("=" * 80)

test_8_passed = True
pytorch_available = False
lightning_available = False

try:
    # Test 1: PyTorch availability
    print("\n1. Testing PyTorch installation...")
    import torch
    import torch.nn as nn
    import torch.optim as optim
    
    pytorch_available = True
    print(f"   ✅ PyTorch: {torch.__version__}")
    print(f"   ✅ CUDA available: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"   ✅ GPU: {torch.cuda.get_device_name(0)}")
    
    # Test 2: Simple PyTorch model training
    print("\n2. Testing PyTorch model training...")
    
    # Create simple dataset
    X_train = torch.randn(1000, 10)
    y_train = torch.randint(0, 2, (1000,))
    
    # Define model
    class SimpleNet(nn.Module):
        def __init__(self):
            super(SimpleNet, self).__init__()
            self.fc1 = nn.Linear(10, 32)
            self.fc2 = nn.Linear(32, 2)
            self.relu = nn.ReLU()
        
        def forward(self, x):
            x = self.relu(self.fc1(x))
            x = self.fc2(x)
            return x
    
    model = SimpleNet()
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    
    # Train for 5 epochs
    model.train()
    for epoch in range(5):
        optimizer.zero_grad()
        outputs = model(X_train)
        loss = criterion(outputs, y_train)
        loss.backward()
        optimizer.step()
    
    print(f"   ✅ Model trained successfully")
    print(f"   ✅ Final loss: {loss.item():.4f}")
    
    # Test 3: PyTorch Lightning (optional but recommended)
    print("\n3. Testing PyTorch Lightning (optional)...")
    try:
        import pytorch_lightning as pl
        lightning_available = True
        print(f"   ✅ PyTorch Lightning: {pl.__version__}")
        print(f"   ✅ Distributed training support available")
    except ImportError:
        print(f"   ⚠️  PyTorch Lightning not installed (optional)")
        print(f"   ℹ️  For distributed deep learning, install: pip install pytorch-lightning")
    
    # Test 4: Spark → PyTorch integration
    print("\n4. Testing Spark → PyTorch data pipeline...")
    
    # Convert Spark DataFrame to PyTorch tensors
    spark_df = spark.range(1000).withColumn("feature", F.rand())
    pandas_df = spark_df.select("feature").toPandas()
    
    # Convert to PyTorch tensor
    torch_tensor = torch.tensor(pandas_df.values, dtype=torch.float32)
    print(f"   ✅ Spark → Pandas → PyTorch conversion successful")
    print(f"   ✅ Tensor shape: {torch_tensor.shape}")
    
    print("\n" + "=" * 80)
    print("✅ TEST 8 PASSED: PyTorch and deep learning ready")
    print("=" * 80)
    
except ImportError as e:
    test_8_passed = False
    pytorch_available = False
    print(f"\n❌ TEST 8 FAILED: PyTorch not installed")
    print(f"   Error: {str(e)}")
    print(f"   Required: Install PyTorch (included in Databricks ML Runtime)")
    print("=" * 80)
except Exception as e:
    test_8_passed = False
    print(f"\n❌ TEST 8 FAILED: {str(e)}")
    print("=" * 80)

## Test 9: DBFS Storage Access

In [ ]:
print("=" * 80)
print("TEST 9: DBFS STORAGE ACCESS")
print("=" * 80)

test_9_passed = False
# DBFS path for shared datasets (works on all cluster types)
SHARED_DATA_PATH = "/dbfs/shared_datasets"

try:
    import time
    
    print("\n1. Verifying DBFS access...")
    print(f"   Storage path: {SHARED_DATA_PATH}")
    
    # Test file operations with unique directory name (avoid collisions)
    run_id = str(int(time.time() * 1000))  # Millisecond timestamp for uniqueness
    test_dir = f"{SHARED_DATA_PATH}/_validation_test_{run_id}"
    
    print(f"\n2. Testing file operations...")
    
    # Create directory (this validates DBFS access)
    try:
        dbutils.fs.mkdirs(test_dir)
        print(f"   ✅ mkdir: Created test directory")
        print(f"   ✅ DBFS is accessible and writable")
    except Exception as e:
        raise Exception(f"Cannot create directory in DBFS: {str(e)[:200]}")
    
    # Write file
    test_file = f"{test_dir}/test.txt"
    dbutils.fs.put(test_file, "Hello from DBFS validation!", overwrite=True)
    print(f"   ✅ write: Created test file")
    
    # Read file
    content = dbutils.fs.head(test_file)
    if "validation" in content:
        print(f"   ✅ read: File content verified")
    else:
        raise Exception("File content mismatch")
    
    # List files
    files = dbutils.fs.ls(test_dir)
    if len(files) > 0:
        print(f"   ✅ list: Found {len(files)} file(s)")
    else:
        raise Exception("File listing failed")
    
    # Delete test files
    dbutils.fs.rm(test_dir, recurse=True)
    print(f"   ✅ delete: Cleaned up test directory")
    
    test_9_passed = True
    
    print("\n" + "=" * 80)
    print("✅ TEST 9 PASSED: DBFS storage access validated")
    print(f"   Storage Path: {SHARED_DATA_PATH}")
    print(f"   All file operations work correctly")
    print("=" * 80)
    
except Exception as e:
    print(f"\n❌ TEST 9 FAILED: {str(e)}")
    print("\nTroubleshooting:")
    print("  1. Verify cluster has DBFS access enabled")
    print("  2. Check cluster permissions")
    print("  3. Ensure sufficient disk space")
    print("=" * 80)

## Test 10: Dataset Download & Storage

In [ ]:
print("=" * 80)
print("TEST 10: DATASET DOWNLOAD & STORAGE")
print("=" * 80)

test_10_passed = False

try:
    if not test_9_passed:
        raise Exception("Test 9 must pass first (storage access required)")
    
    import urllib.request
    import os
    import time
    
    # Create unique directory for this validation run
    run_id = str(int(time.time()))
    test_dataset_dir = f"{SHARED_DATA_PATH}/test_datasets_{run_id}"
    
    print("\n1. Testing small file download (verification)...")
    
    # Test with small CSV file first (~10KB)
    small_url = "https://people.sc.fsu.edu/~jburkardt/data/csv/airtravel.csv"
    local_temp = "/tmp/test_download.csv"
    
    print(f"   Downloading from: {small_url}")
    urllib.request.urlretrieve(small_url, local_temp)
    file_size = os.path.getsize(local_temp) / 1024  # KB
    print(f"   ✅ Downloaded to temp: {file_size:.1f} KB")
    
    # Copy to DBFS
    dbfs_csv = f"{test_dataset_dir}/test_airtravel.csv"
    dbutils.fs.mkdirs(test_dataset_dir)
    dbutils.fs.cp(f"file:{local_temp}", dbfs_csv)
    print(f"   ✅ Copied to DBFS")
    
    # Verify with Spark
    verify_df = spark.read.csv(dbfs_csv, header=True)
    csv_rows = verify_df.count()
    print(f"   ✅ Loaded with Spark: {csv_rows} rows")
    
    # Cleanup temp
    os.remove(local_temp)
    
    print("\n2. Testing NYC Taxi dataset download (Week 3)...")
    print("   ⚠️  This downloads ~45MB and may take 30-90 seconds")
    
    # NYC Taxi official source
    taxi_url = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.parquet"
    local_taxi = "/tmp/yellow_tripdata_2024-01.parquet"
    
    try:
        print(f"   📥 Downloading from: {taxi_url}")
        print(f"   ⏳ Please wait...")
        
        # Download
        urllib.request.urlretrieve(taxi_url, local_taxi)
        taxi_size_mb = os.path.getsize(local_taxi) / (1024 ** 2)
        print(f"   ✅ Downloaded: {taxi_size_mb:.1f} MB")
        
        # Copy to DBFS
        dbfs_taxi = f"{test_dataset_dir}/yellow_tripdata_2024-01.parquet"
        dbutils.fs.cp(f"file:{local_taxi}", dbfs_taxi)
        print(f"   ✅ Copied to DBFS")
        
        # Verify with Spark
        taxi_df = spark.read.parquet(dbfs_taxi)
        taxi_rows = taxi_df.count()
        taxi_cols = len(taxi_df.columns)
        print(f"   ✅ Verified: {taxi_rows:,} rows, {taxi_cols} columns")
        
        # Show sample
        print(f"\n   Sample schema:")
        for col in taxi_df.columns[:5]:
            print(f"      - {col}")
        
        # Cleanup temp
        os.remove(local_taxi)
        taxi_download_works = True
        
    except Exception as e:
        print(f"   ❌ NYC Taxi download failed: {str(e)[:200]}")
        print(f"   ℹ️  This may be due to network restrictions or firewall")
        taxi_download_works = False
    
    print("\n3. Testing UCI dataset download (Week 4)...")
    
    try:
        # Install ucimlrepo if not already installed
        import sys
        import subprocess
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "ucimlrepo"])
        
        from ucimlrepo import fetch_ucirepo
        import pandas as pd
        
        # Fetch Iranian Churn Dataset (ID: 563) - Small dataset
        print(f"   📥 Fetching UCI Iranian Churn Dataset (ID: 563)...")
        churn_dataset = fetch_ucirepo(id=563)
        
        # Combine features and targets
        X = churn_dataset.data.features
        y = churn_dataset.data.targets
        churn_pandas = pd.concat([X, y], axis=1)
        
        print(f"   ✅ Fetched: {len(churn_pandas)} rows, {len(churn_pandas.columns)} columns")
        
        # Convert to Spark DataFrame
        churn_spark = spark.createDataFrame(churn_pandas)
        
        # Save to DBFS
        churn_path = f"{test_dataset_dir}/telecom_churn.parquet"
        churn_spark.write.mode("overwrite").parquet(churn_path)
        print(f"   ✅ Saved to DBFS")
        
        # Verify
        verify_churn = spark.read.parquet(churn_path)
        churn_rows = verify_churn.count()
        print(f"   ✅ Verified: {churn_rows:,} rows loaded from DBFS")
        
        uci_download_works = True
        
    except Exception as e:
        print(f"   ❌ UCI download failed: {str(e)[:200]}")
        print(f"   ℹ️  You may need to install ucimlrepo manually")
        uci_download_works = False
    
    # Cleanup all test datasets
    print(f"\n4. Cleaning up test files...")
    dbutils.fs.rm(test_dataset_dir, recurse=True)
    print(f"   ✅ Removed test directory")
    
    # Determine overall pass/fail
    if taxi_download_works and uci_download_works:
        test_10_passed = True
        print("\n" + "=" * 80)
        print("✅ TEST 10 PASSED: All dataset downloads work")
        print("   ✅ NYC Taxi dataset: Download and Spark read successful")
        print("   ✅ UCI Churn dataset: Download and Spark read successful")
        print("   ℹ️  Week 3/4 notebooks will download datasets on-the-fly")
        print("=" * 80)
    else:
        print("\n" + "=" * 80)
        print("⚠️  TEST 10 PARTIAL PASS: Some downloads failed")
        if taxi_download_works:
            print("   ✅ NYC Taxi dataset works")
        else:
            print("   ❌ NYC Taxi dataset failed - check network/firewall")
        if uci_download_works:
            print("   ✅ UCI Churn dataset works")
        else:
            print("   ❌ UCI Churn dataset failed - may need manual setup")
        print("=" * 80)
    
except Exception as e:
    print(f"\n❌ TEST 10 FAILED: {str(e)}")
    print("\nTroubleshooting:")
    print("  1. Check internet connectivity")
    print("  2. Verify firewall allows HTTPS downloads")
    print("  3. Ensure DBFS has sufficient space (~100MB)")
    print("=" * 80)

## Test 11: Package Installation (%pip)

In [ ]:
print("=" * 80)
print("TEST 11: PACKAGE INSTALLATION (%pip)")
print("=" * 80)

test_11_passed = False

try:
    import sys
    import subprocess
    
    print("\n1. Testing %pip install (notebook-scoped)...")
    print("   Installing ucimlrepo package...")
    
    # Install package quietly
    result = subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "ucimlrepo"])
    print(f"   ✅ Package installation completed")
    
    print("\n2. Verifying package import...")
    
    # Verify import works
    try:
        import ucimlrepo
        print(f"   ✅ Import successful: ucimlrepo {ucimlrepo.__version__}")
    except ImportError as e:
        raise Exception(f"Package installed but import failed: {e}")
    
    print("\n3. Testing package functionality...")
    
    # Test basic functionality
    try:
        # This should work without errors
        dataset_info = ucimlrepo.list_available_datasets()
        print(f"   ✅ Package functions correctly")
    except Exception as e:
        raise Exception(f"Package function test failed: {str(e)[:100]}")
    
    test_11_passed = True
    
    print("\n" + "=" * 80)
    print("✅ TEST 11 PASSED: Package installation works")
    print("   ℹ️  Note: Packages are notebook-scoped and non-persistent")
    print("   ℹ️  Students must reinstall at start of each session")
    print("=" * 80)
    
except Exception as e:
    print(f"\n❌ TEST 11 FAILED: {str(e)}")
    print("\nTroubleshooting:")
    print("  1. Verify cluster has internet access")
    print("  2. Check if pip is available")
    print("  3. Try manual install: %pip install ucimlrepo")
    print("=" * 80)

## Test 12: Model Persistence

In [ ]:
print("=" * 80)
print("TEST 12: MODEL PERSISTENCE")
print("=" * 80)

test_12_passed = False

try:
    if not test_9_passed:
        raise Exception("Test 9 must pass first (storage access required)")
    
    from pyspark.ml.regression import LinearRegression
    import time
    
    print("\n1. Training simple Linear Regression model...")
    
    # Create simple training data (use existing from Test 4 if available)
    if 'train_df' not in locals():
        sample_data = spark.range(1000).withColumn("feature", F.rand()).withColumn("label", F.rand() * 10)
        from pyspark.ml.feature import VectorAssembler
        assembler = VectorAssembler(inputCols=["feature"], outputCol="features")
        train_sample = assembler.transform(sample_data)
    else:
        train_sample = train_df.limit(1000)
    
    # Train model
    lr_test = LinearRegression(featuresCol="features" if 'features' in train_sample.columns else "scaled_features", 
                               labelCol="label", maxIter=5)
    model_test = lr_test.fit(train_sample)
    print(f"   ✅ Model trained")
    
    print("\n2. Saving model to DBFS...")
    
    # Create unique model path
    run_id = str(int(time.time()))
    model_path = f"{SHARED_DATA_PATH}/test_models/lr_model_{run_id}"
    
    try:
        # Try MLflow approach (recommended)
        import mlflow
        # Convert DBFS path to local path for MLflow
        local_model_path = model_path.replace("/dbfs", "")
        mlflow.spark.save_model(model_test, local_model_path)
        print(f"   ✅ Model saved with MLflow")
        use_mlflow = True
    except Exception as mlflow_error:
        # Fallback: Direct MLlib save
        print(f"   ⚠️  MLflow not available, using direct save")
        model_test.save(model_path)
        print(f"   ✅ Model saved with MLlib")
        use_mlflow = False
    
    print("\n3. Loading model from DBFS...")
    
    try:
        if use_mlflow:
            loaded_model = mlflow.spark.load_model(local_model_path)
        else:
            from pyspark.ml.regression import LinearRegressionModel
            loaded_model = LinearRegressionModel.load(model_path)
        
        print(f"   ✅ Model loaded successfully")
    except Exception as e:
        raise Exception(f"Model load failed: {str(e)[:200]}")
    
    print("\n4. Verifying loaded model works...")
    
    # Make predictions
    predictions = loaded_model.transform(train_sample.limit(10))
    pred_count = predictions.select("prediction").count()
    
    if pred_count > 0:
        print(f"   ✅ Model predictions work: {pred_count} rows")
    else:
        raise Exception("Model produced no predictions")
    
    # Cleanup
    print("\n5. Cleaning up test model...")
    dbutils.fs.rm(f"{SHARED_DATA_PATH}/test_models", recurse=True)
    print(f"   ✅ Removed test model")
    
    test_12_passed = True
    
    print("\n" + "=" * 80)
    print("✅ TEST 12 PASSED: Model persistence works")
    if use_mlflow:
        print("   ✅ MLflow model save/load successful")
    else:
        print("   ✅ MLlib model save/load successful")
    print("=" * 80)
    
except Exception as e:
    print(f"\n❌ TEST 12 FAILED: {str(e)}")
    print("\nTroubleshooting:")
    print("  1. Verify MLflow is available in runtime")
    print("  2. Check DBFS write permissions")
    print("  3. Ensure sufficient disk space")
    print("=" * 80)

## Test 13: DataFrame Export (Parquet/CSV)

In [ ]:
print("=" * 80)
print("TEST 13: DATAFRAME EXPORT (PARQUET/CSV)")
print("=" * 80)

test_13_passed = False

try:
    if not test_9_passed:
        raise Exception("Test 9 must pass first (storage access required)")
    
    import time
    
    # Create test DataFrame
    print("\n1. Creating test DataFrame...")
    test_data = spark.range(100).withColumn("value", F.rand() * 100)
    print(f"   ✅ Created DataFrame: {test_data.count()} rows")
    
    run_id = str(int(time.time()))
    test_export_dir = f"{SHARED_DATA_PATH}/test_exports_{run_id}"
    
    # Test Parquet export
    print("\n2. Testing Parquet export...")
    parquet_path = f"{test_export_dir}/data.parquet"
    
    test_data.write.mode("overwrite").parquet(parquet_path)
    print(f"   ✅ Exported to Parquet")
    
    # Verify Parquet read
    verify_parquet = spark.read.parquet(parquet_path)
    parquet_rows = verify_parquet.count()
    print(f"   ✅ Verified Parquet read: {parquet_rows} rows")
    
    # Test CSV export
    print("\n3. Testing CSV export...")
    csv_path = f"{test_export_dir}/data.csv"
    
    test_data.write.mode("overwrite").option("header", "true").csv(csv_path)
    print(f"   ✅ Exported to CSV")
    
    # Verify CSV read
    verify_csv = spark.read.option("header", "true").csv(csv_path)
    csv_rows = verify_csv.count()
    print(f"   ✅ Verified CSV read: {csv_rows} rows")
    
    # Test JSON export (bonus)
    print("\n4. Testing JSON export...")
    json_path = f"{test_export_dir}/data.json"
    
    test_data.limit(10).write.mode("overwrite").json(json_path)
    print(f"   ✅ Exported to JSON")
    
    # Verify JSON read
    verify_json = spark.read.json(json_path)
    json_rows = verify_json.count()
    print(f"   ✅ Verified JSON read: {json_rows} rows")
    
    # Cleanup
    print("\n5. Cleaning up test files...")
    dbutils.fs.rm(test_export_dir, recurse=True)
    print(f"   ✅ Removed test exports")
    
    test_13_passed = True
    
    print("\n" + "=" * 80)
    print("✅ TEST 13 PASSED: DataFrame export works")
    print("   ✅ Parquet: Write and read successful")
    print("   ✅ CSV: Write and read successful")
    print("   ✅ JSON: Write and read successful")
    print("=" * 80)
    
except Exception as e:
    print(f"\n❌ TEST 13 FAILED: {str(e)}")
    print("\nTroubleshooting:")
    print("  1. Check DBFS write permissions")
    print("  2. Verify sufficient disk space")
    print("  3. Ensure DataFrame is not corrupted")
    print("=" * 80)

## Test 14: Plot & Artifact Storage

In [ ]:
print("=" * 80)
print("TEST 14: PLOT & ARTIFACT STORAGE")
print("=" * 80)

test_14_passed = False

try:
    if not test_9_passed:
        raise Exception("Test 9 must pass first (storage access required)")
    
    import matplotlib.pyplot as plt
    import time
    import os
    
    # Configure matplotlib for headless mode (no display)
    plt.switch_backend('Agg')
    
    print("\n1. Creating test plot...")
    
    # Create simple plot
    fig, ax = plt.subplots(figsize=(8, 6))
    x = [1, 2, 3, 4, 5]
    y = [1, 4, 9, 16, 25]
    ax.plot(x, y, marker='o', linestyle='-', color='b')
    ax.set_title('Test Plot - Databricks Validation')
    ax.set_xlabel('X axis')
    ax.set_ylabel('Y axis')
    ax.grid(True)
    
    print(f"   ✅ Plot created")
    
    # Save to DBFS
    print("\n2. Saving plot to DBFS...")
    
    run_id = str(int(time.time()))
    plot_dir_dbfs = f"{SHARED_DATA_PATH}/test_plots_{run_id}"
    
    # Create directory in DBFS first
    dbutils.fs.mkdirs(plot_dir_dbfs)
    
    # Convert to local file system path for matplotlib
    plot_dir_local = plot_dir_dbfs.replace("/dbfs", "")
    
    # Ensure local directory exists (matplotlib needs this)
    os.makedirs(plot_dir_local, exist_ok=True)
    
    # Save as PNG
    png_path = f"{plot_dir_local}/test_plot.png"
    plt.savefig(png_path, dpi=100, bbox_inches='tight')
    print(f"   ✅ Saved PNG")
    
    # Close plot
    plt.close()
    
    # Verify file exists in DBFS
    print("\n3. Verifying saved plot...")
    files = dbutils.fs.ls(plot_dir_dbfs)
    png_files = [f for f in files if f.name.endswith('.png')]
    
    if len(png_files) > 0:
        file_size_kb = png_files[0].size / 1024
        print(f"   ✅ Plot file found: {png_files[0].name} ({file_size_kb:.1f} KB)")
    else:
        raise Exception("Plot file not found in DBFS")
    
    # Test saving additional formats
    print("\n4. Testing additional image formats...")
    
    # Create another plot
    fig2, ax2 = plt.subplots()
    ax2.bar(['A', 'B', 'C'], [10, 20, 15])
    ax2.set_title('Bar Chart Test')
    
    # Save as PDF
    pdf_path = f"{plot_dir_local}/test_plot.pdf"
    plt.savefig(pdf_path, format='pdf')
    print(f"   ✅ Saved PDF")
    
    plt.close()
    
    # Verify both files
    all_files = dbutils.fs.ls(plot_dir_dbfs)
    print(f"   ✅ Total artifacts saved: {len(all_files)}")
    for f in all_files:
        print(f"      - {f.name} ({f.size / 1024:.1f} KB)")
    
    # Cleanup
    print("\n5. Cleaning up test plots...")
    dbutils.fs.rm(plot_dir_dbfs, recurse=True)
    print(f"   ✅ Removed test plots")
    
    test_14_passed = True
    
    print("\n" + "=" * 80)
    print("✅ TEST 14 PASSED: Plot and artifact storage works")
    print("   ✅ PNG export: Successful")
    print("   ✅ PDF export: Successful")
    print("   ✅ File verification: All artifacts saved correctly")
    print("=" * 80)
    
except Exception as e:
    print(f"\n❌ TEST 14 FAILED: {str(e)}")
    print("\nTroubleshooting:")
    print("  1. Verify matplotlib is installed")
    print("  2. Check DBFS write permissions")
    print("  3. Ensure headless backend is configured")
    print("=" * 80)

---

## Final Summary & Recommendations

In [ ]:
print("\n\n")
print("=" * 80)
print("=" * 80)
print("FINAL VALIDATION SUMMARY")
print("=" * 80)
print("=" * 80)

# Collect results
tests = [
    ("Test 1: Cluster Configuration", True),  # Always passes if notebook runs
    ("Test 2: Distributed Data Operations", test_2_passed),
    ("Test 3: Feature Engineering", test_3_passed),
    ("Test 4: Regression (Week 3)", test_4_passed),
    ("Test 5: Classification (Week 4)", test_5_passed),
    ("Test 6: Required Libraries", test_6_passed),
    ("Test 7: Multi-User Concurrency", test_7_passed),
    ("Test 8: PyTorch & Deep Learning", test_8_passed),
    ("Test 9: DBFS Storage Access", test_9_passed),
    ("Test 10: Dataset Download & Storage", test_10_passed),
    ("Test 11: Package Installation (%pip)", test_11_passed),
    ("Test 12: Model Persistence", test_12_passed),
    ("Test 13: DataFrame Export (Parquet/CSV)", test_13_passed),
    ("Test 14: Plot & Artifact Storage", test_14_passed),
]

passed = sum(1 for _, result in tests if result)
total = len(tests)

print(f"\nTests Passed: {passed}/{total}\n")
print("-" * 80)

for test_name, result in tests:
    status = "✅ PASS" if result else "❌ FAIL"
    print(f"{status}  {test_name}")

print("-" * 80)

# Overall verdict
print("\n" + "=" * 80)
if passed == total:
    print("🎉 ALL TESTS PASSED!")
    print("=" * 80)
    print("\n✅ YOUR CLUSTER IS READY FOR WEEKS 3 & 4")
    print("\nYour Databricks cluster is properly configured for:")
    print("  • Week 3: Spark ML Regression")
    print("  • Week 4: Spark ML Classification")
    print("  • Deep Learning: PyTorch training")
    if pytorch_available and lightning_available:
        print("  • Distributed Deep Learning: PyTorch Lightning available")
    print("  • DBFS Storage: Accessible and writable")
    print("  • Dataset Downloads: NYC Taxi and UCI datasets work")
    print("  • All 20 students can work concurrently")
    print("\nCluster Type: Shared (Unity Catalog)")
    print("Runtime: Databricks ML LTS 17.3 (Spark 4.0.0, Python 3.12.3)")
    print(f"Storage Path: {SHARED_DATA_PATH}")
    print("\n📝 IMPORTANT NOTES:")
    print("   • Datasets are downloaded on-the-fly in Week 3/4 notebooks")
    print("   • Data stored in DBFS (Databricks File System)")
    print("   • Notebooks should NOT access spark.sparkContext directly")
    
else:
    print("⚠️  SOME TESTS FAILED")
    print("=" * 80)
    print("\n❌ YOUR CLUSTER NEEDS ADJUSTMENTS")
    print("\nSend this report to your IT team with the following requests:\n")
    
    if not test_2_passed:
        print("❌ Distributed Operations Failed:")
        print("   Request: Verify cluster has workers attached and is running")
        print("   Required: Multi-node cluster with at least 2 workers\n")
    
    if not test_3_passed:
        print("❌ Feature Engineering Failed:")
        print("   Request: Verify Databricks ML Runtime is installed")
        print("   Required: Databricks Runtime 17.3 LTS ML or higher\n")
    
    if not test_4_passed:
        print("❌ Regression Models Failed:")
        print("   Request: Verify Spark MLlib is available")
        print("   Required: Full MLlib library access\n")
    
    if not test_5_passed:
        print("❌ Classification Models Failed:")
        print("   Request: Verify Spark MLlib classification libraries")
        print("   Required: Full MLlib library access\n")
    
    if not test_6_passed:
        print(f"❌ Missing Libraries: {', '.join(missing_libs)}")
        print("   Request: Install missing Python libraries")
        print("   Recommended: Use Databricks ML Runtime (includes all libraries)\n")
    
    if not test_7_passed:
        print("❌ Concurrency Issues:")
        print("   Request: Verify cluster has Fair Scheduler enabled")
        print("   Required: spark.scheduler.mode = FAIR\n")
    
    if not test_8_passed:
        print("❌ PyTorch/Deep Learning Failed:")
        print("   Request: Install PyTorch in Databricks ML Runtime")
        print("   Required: PyTorch 2.7+ (included in Databricks ML Runtime 17.3)")
        if not lightning_available:
            print("   Optional: Install pytorch-lightning for distributed training\n")
    
    if not test_9_passed:
        print("❌ DBFS Storage Access Failed:")
        print("   Request: Verify DBFS is enabled and accessible")
        print("   Required: DBFS read/write permissions\n")
    
    if not test_10_passed:
        print("❌ Dataset Download Failed:")
        print("   Request: Check network/firewall settings")
        print("   Required: Allow HTTPS downloads from external sources")
        print("   URLs: d37ci6vzurychx.cloudfront.net, UCI ML Repository\n")
    
    if not test_11_passed:
        print("❌ Package Installation Failed:")
        print("   Request: Verify pip and internet access")
        print("   Required: %pip install must work for notebook-scoped packages\n")
    
    if not test_12_passed:
        print("❌ Model Persistence Failed:")
        print("   Request: Check MLflow availability or DBFS write permissions")
        print("   Required: Ability to save/load Spark ML models to DBFS\n")
    
    if not test_13_passed:
        print("❌ DataFrame Export Failed:")
        print("   Request: Verify DBFS write permissions")
        print("   Required: Export DataFrames as Parquet/CSV/JSON\n")
    
    if not test_14_passed:
        print("❌ Plot Storage Failed:")
        print("   Request: Check matplotlib and DBFS access")
        print("   Required: Save plots/artifacts to DBFS\n")

print("\n" + "=" * 80)
print("RECOMMENDED CLUSTER CONFIGURATION")
print("=" * 80)
print("\nFor 20 concurrent students with Spark ML + PyTorch:")
print("\n  Cluster Type: Shared (Unity Catalog)")
print("  Access Mode: Shared")
print("  Runtime: Databricks Runtime 17.3 LTS ML")
print("  Spark: 4.0.0 (DataFrame API fully supported)")
print("  Python: 3.12.3")
print("  Driver: 16 GB RAM (recommended for deep learning)")
print("  Workers: 2-4 workers with 16 GB RAM each")
print("  GPU: Optional (CPU works, GPU speeds up PyTorch training)")
print("  Scheduler: FAIR (for multi-user)")
print("  Autoscaling: Enabled (2-8 workers)")
print("\n  Storage:")
print(f"    Path: {SHARED_DATA_PATH}")
print("    Type: DBFS (Databricks File System)")
print("    Access: Read/Write for all cluster users")
print("\n  Included in ML Runtime 17.3:")
print("    • Spark MLlib 4.0 (regression, classification)")
print("    • PyTorch 2.7.0 (deep learning)")
print("    • scikit-learn 1.6.1, pandas 2.2.3, numpy 2.1.3")
print("    • MLflow for model tracking")
print("\n" + "=" * 80)

print("\n📧 Questions? Contact your Databricks admin or Bread Financial IT team.")
print("\n" + "=" * 80)